# 03 — Data Cleaning

## Projet : AI Business Advisor V2

### Objectif

Cette étape applique les décisions de nettoyage issues du Data Audit
et de l'EDA.

Le nettoyage doit rester :

- reproductible ;
- justifié ;
- traçable ;
- indépendant de la variable cible autant que possible.

Les données originales situées dans `data/raw/` ne seront jamais modifiées.

Les opérations statistiques dépendant du jeu d'entraînement,
comme l'imputation, la standardisation, l'encodage ou certains traitements
des outliers, seront réalisées plus tard dans le pipeline Machine Learning
afin d'éviter le data leakage.

Les datasets nettoyés seront enregistrés dans `data/interim/`.

In [1]:
import pandas as pd 
import numpy as np 

from pathlib import Path

pd.set_option("display.max_columns",None)
pd.set_option("display.max_rows",100)
pd.set_option("display.width",200)

In [2]:
cwd = Path.cwd()

possible_raws_paths = [
    cwd / "data" / "raw",
    cwd / "data-science" / "data" / "raw",
    cwd.parent / "data" / "raw"
]

RAW_DIR = None

for path in possible_raws_paths:
    if path.exists():
        RAW_DIR = path
        break

if RAW_DIR is None:
    raise FileNotFoundError(
        "Impossible de localiser data/raw"
    )

DATA_DIR = RAW_DIR.parent 

INTERIM_DIR = DATA_DIR / "interim"

INTERIM_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("RAW     :",RAW_DIR.resolve())
print("INTERIM  :",INTERIM_DIR.resolve())

RAW     : C:\Users\user\ai-business-advisor-v2\data-science\data\raw
INTERIM  : C:\Users\user\ai-business-advisor-v2\data-science\data\interim


In [3]:
FREQ_PATH = RAW_DIR / "freMTPL2freq.csv"

FRAUD_PATH = RAW_DIR / "fraud_oracle.csv"

freq_raw = pd.read_csv(
    FREQ_PATH
)

fraud_raw = pd.read_csv(
    FRAUD_PATH
)

print(
    "freMTPL2freq : ",
    freq_raw.shape
)

print(
    "Fraud : ",
    fraud_raw.shape
)

freMTPL2freq :  (678013, 12)
Fraud :  (15420, 33)


In [4]:
freq_clean = (
    freq_raw.copy()
)

fraud_clean = (
    fraud_raw.copy()
)

In [5]:
cleaning_log = []

In [6]:
def add_cleaning_log(
        dataset,
        problem,
        decision,
        transformation,
        affected_rows
):

    cleaning_log.append({
        "dataset":dataset,
        "problem" :problem,
        "decision":decision,
        "transformation":transformation,
        "affected_rows":affected_rows
    })

In [7]:
before_summary = pd.DataFrame({
    "datset": [
        "freMTPL2freq",
        "fraud_oracle"
    ],
    "rows": [
        len(freq_clean),
        len(fraud_clean)
    ],
    "colmuns": [
        freq_clean.shape[1],
        fraud_clean.shape[1]
    ],
    "missing_values": [
        freq_clean.isna().sum().sum(),
        fraud_clean.isna().sum().sum()
    ],
    "duplicates":[
        freq_clean.duplicated().sum(),
        fraud_clean.duplicated().sum()
    ]
})

before_summary

,datset,rows,colmuns,missing_values,duplicates
0,freMTPL2freq,678013,12,0,0
1,fraud_oracle,15420,33,0,0


# Partie A — Cleaning freMTPL2freq

Les principales anomalies détectées lors de l'audit sont :

- `IDpol` stocké en float alors qu'il représente un identifiant ;
- certaines valeurs `Exposure > 1` ;
- quelques valeurs très élevées de `ClaimNb` ;
- des valeurs extrêmes pour `VehAge`, `DrivAge`, `BonusMalus` et `Density`.

Toutes les valeurs extrêmes ne seront pas automatiquement supprimées.

Une valeur extrême n'est pas nécessairement une valeur incorrecte.

In [8]:
freq_clean["IDpol"].dtype

fractional_ids = (
    (
    freq_clean["IDpol"]
    % 1
) != 0
).sum()

print("IDpol avec partie décimale") 

freq_clean["IDpol"] = (
    freq_clean["IDpol"]
    .astype("int64")
)

add_cleaning_log(
    dataset="freMTPL2freq",
    problem="IDpol stocké en float64",
    decision="Conversion en identifiant unique",
    transformation="float64 -> int64",
    affected_rows=len(freq_clean)
)

assert(
    freq_clean["IDpol"]
    .is_unique
)

print("IDpol unique : OK")

IDpol avec partie décimale
IDpol unique : OK


In [9]:
freq_cat_cols = [
    "Area",
    "VehBrand",
    "VehGas",
    "Region"
]

for col in freq_cat_cols:

    freq_clean[col] = (
        freq_clean[col]
        .astype("string")
        .str.strip()
    )

invalid_exposure = (
    freq_clean["Exposure"] > 1
)

invalid_exposure.sum()
invalid_exposure.mean() * 100

np.float64(0.18052751200935677)

In [10]:
(freq_clean["Exposure"] <= 0).sum()

np.int64(0)

In [11]:
rows_before_exposure = len(
    freq_clean
)

freq_clean = (
    freq_clean.loc[
        (freq_clean["Exposure"] > 0)
        &
        (freq_clean["Exposure"] <= 1)
    ]
    .copy()
)
removed_exposure = (
    rows_before_exposure
    -
    len(freq_clean)
)

print(" Lignes exclues : ", removed_exposure)

add_cleaning_log(
    dataset="freMTLP2freq",
    problem="Exposure > 1",
    decision="Population de modèlisation limitée à 0< Exposure <=1",
    transformation="Exclusion des observations Exposure > 1 ",
    affected_rows=removed_exposure 
)

raw_occurence = (
    (freq_raw["ClaimNb"] > 0)
    .mean()
)

raw_frequency = (
    freq_raw["ClaimNb"].sum()
    /
    freq_raw["Exposure"].sum()
)

clean_occurence = (
    (freq_clean["ClaimNb"] >0)
    .mean()
)
clean_frequency = (
    freq_clean["ClaimNb"].sum()
    /
    freq_clean["Exposure"].sum()
)

print("Occurence RAW : ",raw_occurence)
print("Frequency RAW :",clean_occurence)
print()
print("Frequency RAW :", raw_frequency)
print("Frequency CLEAN : ",clean_frequency)


 Lignes exclues :  1224
Occurence RAW :  0.05023502499214617
Frequency RAW : 0.0502505212111899

Frequency RAW : 0.1007030846404707
Frequency CLEAN :  0.10093630817105106


In [12]:
(freq_clean["ClaimNb"]>4).sum()

claim_extreme_count = (
    freq_clean["ClaimNb"] > 4
).sum()

print("ClaimNb > 4 :",claim_extreme_count)

add_cleaning_log(
    dataset="freMTPL2freq",
    problem="ClaimNb > 4 pour quelques observations",
    decision="Conserver les valeurs",
    transformation="Aucune",
    affected_rows=claim_extreme_count
)

ClaimNb > 4 : 9


### Décision — ClaimNb

Les valeurs élevées de `ClaimNb` sont rares mais correspondent à la variable cible.

Les supprimer uniquement parce qu'elles sont extrêmes pourrait introduire
un biais en éliminant les assurés présentant le plus grand nombre de sinistres.

Elles sont donc conservées tant qu'aucune preuve ne montre qu'elles sont erronées.

In [13]:
veh_age_100 = (
    freq_clean["VehAge"]==100
).sum()
veh_age_100

add_cleaning_log(
    dataset="freMTPL2freq",
    problem="VehAge atteint 100",
    decision="Valeur extreme conservée en absence de preuve d'erreur",
    transformation="Aucune",
    affected_rows=veh_age_100
)

driv_age_100 = (
    freq_clean["DrivAge"] == 100
).sum()

print(
    driv_age_100
)

add_cleaning_log(
    dataset="freMTPL2freq",
    problem="DrivAge atteint 100",
    decision="Valeur rare mais non automatiquement supprimée",
    transformation="Aucune",
    affected_rows=driv_age_100
)

3


In [14]:
freq_clean["BonusMalus"].describe(
    percentiles=[
        0.95,
        0.99
    ]
)

high_bonus = (
    freq_clean["BonusMalus"]>150
).sum()

high_bonus

add_cleaning_log(
    dataset="freMTPL2freq",
    problem="BonusMalus élevé",
    decision="Conserver car valeur métier potentiellement informative",
    transformation="Aucune",
    affected_rows=high_bonus
)
density_max_count = (
    freq_clean["Density"] == 27000
).sum()

density_max_count


np.int64(10500)

In [15]:
assert freq_clean["IDpol"].is_unique

assert(
    freq_clean["Exposure"] > 0
).all()

assert(
    freq_clean["Exposure"] <= 1
).all()

assert(
    freq_clean["ClaimNb"] >= 0
).all()

assert(
    freq_clean["VehAge"] >= 0
).all()

assert(
    freq_clean["DrivAge"] >= 0
).all()

assert(
    freq_clean["Density"] > 0
).all()

print(
    "Controles freMTPL2freq  : OK"
)

Controles freMTPL2freq  : OK


# Partie B — Cleaning Vehicle Claim Fraud

Les principales anomalies détectées sont :

- `Age = 0` pour 320 observations ;
- une observation possède `MonthClaimed = "0"` ;
- la même observation possède `DayOfWeekClaimed = "0"` ;
- plusieurs variables utilisent la catégorie `"none"` ;
- certaines marques semblent contenir des fautes d'orthographe ;
- certaines variables numériques sont en réalité catégorielles ou ordinales.

Le nettoyage doit éviter de transformer artificiellement
des catégories dont la signification métier n'est pas certaine.

In [16]:
fraud_object_cols = (
    fraud_clean
    .select_dtypes(
        include=["object","string"]
    )
    .columns
)

for col in fraud_object_cols:

    fraud_clean[col] = (
        fraud_clean[col]
        .astype("string")
        .str.strip()
    )

age_zero_mask = (
    fraud_clean["Age"] == 0
)

age_zero_count = (
    age_zero_mask.sum()
)

print(
    "Age = 0 :",
    age_zero_count
)

fraud_clean.loc[
    age_zero_mask,
    "AgeOfPolicyHolder"
].value_counts()

fraud_clean.loc[
    fraud_clean["Age"] == 0,
    "Age"
] = pd.NA

fraud_clean["Age"]= (
    fraud_clean["Age"]
    .astype("Int64")
)

add_cleaning_log(
    dataset="fraud_oracle",
    problem="Age == 0 alors que AgeOFPolicyHolder = 16 to 17",
    decision="Considérer Age =0 comme valeur exacte inconnue",
    transformation="0 -> NA",
    affected_rows=age_zero_count
)

fraud_clean.loc[
    fraud_clean["MonthClaimed"] == "0",
    [
        "PolicyNumber",
        "MonthClaimed",
        "DayOfWeekClaimed"

    ]
]

Age = 0 : 320


,PolicyNumber,MonthClaimed,DayOfWeekClaimed
1516,1517,0,0


In [17]:
invalid_month_count = (
    fraud_clean["MonthClaimed"] == "0"
).sum()

invalid_day_count = (
    fraud_clean["DayOfWeekClaimed"] == "0"
).sum()

fraud_clean.loc[
    fraud_clean["MonthClaimed"] == "0",
    "MonthClaimed"
] = pd.NA

fraud_clean.loc[
    fraud_clean["DayOfWeekClaimed"] == "0",
    "DayOfWeekClaimed"
] = pd.NA

add_cleaning_log(
    dataset="fraud_oracle",
    problem="MonthClaimed = 0",
    decision="Valeur calendaire impossible",
    transformation="0 -> NA",
    affected_rows=invalid_month_count
)


add_cleaning_log(
    dataset="fraud_oracle",
    problem="DayOfWeekClaimed = 0",
    decision="Valeur calendaire impossible",
    transformation="0 -> NA",
    affected_rows=invalid_month_count
)



In [18]:
none_columns = [
    "Days_Policy_Accident",
    "Days_Policy_Claim",
    "PastNumberOfClaims",
    "NumberOfSuppliments"
]

for col in none_columns:
    count= (
        fraud_clean[col]
        .str.lower()
        .eq("none")
        .sum()
    )

print(
    f"{col:25s} : {count}"
)

NumberOfSuppliments       : 7047


In [19]:
for col in none_columns:

    count = (
        fraud_clean[col]
        .str.lower()
        .eq("none")
        .sum()
    )

    add_cleaning_log(
        dataset="vehicle_claim_fraud",
        problem=f'{col} contient "none"',
        decision="Conserver comme catégorie jusqu'à validation sémantique",
        transformation="Aucune",
        affected_rows=count
    )

In [20]:
make_mapping = {
    "Accura": "Acura",
    "Nisson": "Nissan",
    "Porche": "Porsche",
    "Mecedes": "Mercedes"
}
make_typo_count = (
    fraud_clean["Make"]
    .isin(make_mapping.keys())
    .sum()
)
add_cleaning_log(
    dataset="fraud_oracle",
    problem="Nomenclature incohérente pour certaines marques",
    decision="Normalisation orthographique",
    transformation="Accura→Acura, Nisson→Nissan, Porche→Porsche, Mecedes→Mercedes",
    affected_rows=make_typo_count
)
sorted(
    fraud_clean["Make"]
    .dropna()
    .unique()
)


['Accura',
 'BMW',
 'Chevrolet',
 'Dodge',
 'Ferrari',
 'Ford',
 'Honda',
 'Jaguar',
 'Lexus',
 'Mazda',
 'Mecedes',
 'Mercury',
 'Nisson',
 'Pontiac',
 'Porche',
 'Saab',
 'Saturn',
 'Toyota',
 'VW']

In [21]:
fraud_clean["PolicyNumber"].nunique()

15420

In [22]:
potential_leakage_features = [
    "Fault",
    "PoliceReportFiled",
    "WitnessPresent",
    "NumberOfSuppliments",
    "Days_Policy_Claim",
    "AddressChange_Claim"
]

In [23]:
numeric_validation = pd.DataFrame({
    "variable": [
        "WeekOfMonth",
        "WeekOfMonthClaimed",
        "DriverRating",
        "FraudFound_P",
        "Year"
    ],
    "min": [
        fraud_clean["WeekOfMonth"].min(),
        fraud_clean["WeekOfMonthClaimed"].min(),
        fraud_clean["DriverRating"].min(),
        fraud_clean["FraudFound_P"].min(),
        fraud_clean["Year"].min()
    ],
    "max": [
        fraud_clean["WeekOfMonth"].max(),
        fraud_clean["WeekOfMonthClaimed"].max(),
        fraud_clean["DriverRating"].max(),
        fraud_clean["FraudFound_P"].max(),
        fraud_clean["Year"].max()
    ]
})

numeric_validation

,variable,min,max
0,WeekOfMonth,1,5
1,WeekOfMonthClaimed,1,5
2,DriverRating,1,4
3,FraudFound_P,0,1
4,Year,1994,1996


In [25]:
assert set(
    fraud_clean["FraudFound_P"]
    .dropna()
    .unique()
) <= {0, 1}
assert fraud_clean[
    "WeekOfMonth"
].between(
    1,
    5
).all()
assert fraud_clean[
    "WeekOfMonthClaimed"
].between(
    1,
    5
).all()

assert (
    fraud_clean["PolicyNumber"]
    .is_unique
)

assert not (
    fraud_clean["MonthClaimed"]
    == "0"
).any()

assert not (
    fraud_clean["DayOfWeekClaimed"]
    == "0"
).any()

print(
    "Contrôles Fraud : OK"
)

Contrôles Fraud : OK


In [26]:
fraud_clean["Age"].describe()

count      15100.0
mean     40.700331
std      12.309493
min           16.0
25%           31.0
50%           39.0
75%           49.0
max           80.0
Name: Age, dtype: Float64

In [27]:
fraud_clean["Age"].isna().sum()

np.int64(320)

In [28]:
fraud_before_rate = (
    fraud_raw["FraudFound_P"]
    .mean()
)

fraud_after_rate = (
    fraud_clean["FraudFound_P"]
    .mean()
)

print(
    "Fraud rate avant :",
    fraud_before_rate
)

print(
    "Fraud rate après :",
    fraud_after_rate
)

Fraud rate avant : 0.05985732814526589
Fraud rate après : 0.05985732814526589


In [29]:
cleaning_report = pd.DataFrame(
    cleaning_log
)

cleaning_report

,dataset,problem,decision,transformation,affected_rows
0,freMTPL2freq,IDpol stocké en float64,Conversion en identifiant unique,float64 -> int64,678013
1,freMTLP2freq,Exposure > 1,Population de modèlisation limitée à 0< Exposu...,Exclusion des observations Exposure > 1,1224
2,freMTPL2freq,ClaimNb > 4 pour quelques observations,Conserver les valeurs,Aucune,9
3,freMTPL2freq,VehAge atteint 100,Valeur extreme conservée en absence de preuve ...,Aucune,25
4,freMTPL2freq,DrivAge atteint 100,Valeur rare mais non automatiquement supprimée,Aucune,3
5,freMTPL2freq,BonusMalus élevé,Conserver car valeur métier potentiellement in...,Aucune,209
6,fraud_oracle,Age == 0 alors que AgeOFPolicyHolder = 16 to 17,Considérer Age =0 comme valeur exacte inconnue,0 -> NA,320
7,fraud_oracle,MonthClaimed = 0,Valeur calendaire impossible,0 -> NA,1
8,fraud_oracle,DayOfWeekClaimed = 0,Valeur calendaire impossible,0 -> NA,1
9,vehicle_claim_fraud,"Days_Policy_Accident contient ""none""",Conserver comme catégorie jusqu'à validation s...,Aucune,55


In [31]:
REPORT_DIR = (
    RAW_DIR.parent.parent
    / "reports"
    / "cleaning"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

cleaning_report.to_csv(
    REPORT_DIR
    / "cleaning_decisions.csv",
    index=False
)

In [32]:
comparison = pd.DataFrame({
    "dataset": [
        "freMTPL2freq",
        "vehicle_claim_fraud"
    ],

    "raw_rows": [
        len(freq_raw),
        len(fraud_raw)
    ],

    "clean_rows": [
        len(freq_clean),
        len(fraud_clean)
    ],

    "removed_rows": [
        len(freq_raw) - len(freq_clean),
        len(fraud_raw) - len(fraud_clean)
    ],

    "clean_missing": [
        freq_clean.isna().sum().sum(),
        fraud_clean.isna().sum().sum()
    ]
})

comparison

,dataset,raw_rows,clean_rows,removed_rows,clean_missing
0,freMTPL2freq,678013,676789,1224,0
1,vehicle_claim_fraud,15420,15420,0,322


In [33]:
FREQ_CLEAN_PATH = (
    INTERIM_DIR
    / "freMTPL2freq_clean.csv"
)

freq_clean.to_csv(
    FREQ_CLEAN_PATH,
    index=False
)

print(
    FREQ_CLEAN_PATH
)

c:\Users\user\ai-business-advisor-v2\data-science\data\interim\freMTPL2freq_clean.csv


In [34]:
FRAUD_CLEAN_PATH = (
    INTERIM_DIR
    / "vehicle_claim_fraud_clean.csv"
)

fraud_clean.to_csv(
    FRAUD_CLEAN_PATH,
    index=False
)

print(
    FRAUD_CLEAN_PATH
)

c:\Users\user\ai-business-advisor-v2\data-science\data\interim\vehicle_claim_fraud_clean.csv


In [36]:
freq_check = pd.read_csv(
    FREQ_CLEAN_PATH
)

fraud_check = pd.read_csv(
    FRAUD_CLEAN_PATH
)

print(
    freq_check.shape
)

print(
    fraud_check.shape
)


print(
    freq_check.head()
)

print()

print(
    fraud_check.head()
)

(676789, 12)
(15420, 33)
   IDpol  ClaimNb  Exposure Area  VehPower  VehAge  DrivAge  BonusMalus VehBrand   VehGas  Density Region
0      1        1      0.10    D         5       0       55          50      B12  Regular     1217    R82
1      3        1      0.77    D         5       0       55          50      B12  Regular     1217    R82
2      5        1      0.75    B         6       2       52          50      B12   Diesel       54    R22
3     10        1      0.09    B         7       0       46          50      B12   Diesel       76    R72
4     11        1      0.84    B         7       0       46          50      B12   Diesel       76    R72

  Month  WeekOfMonth  DayOfWeek    Make AccidentArea DayOfWeekClaimed MonthClaimed  WeekOfMonthClaimed     Sex MaritalStatus   Age          Fault         PolicyType VehicleCategory     VehiclePrice  \
0   Dec            5  Wednesday   Honda        Urban          Tuesday          Jan                   1  Female        Single  21.0  Polic

In [37]:
assert (
    len(freq_check)
    == len(freq_clean)
)

assert (
    len(fraud_check)
    == len(fraud_clean)
)

print(
    "Fichiers interim sauvegardés correctement."
)

Fichiers interim sauvegardés correctement.


# Bilan Cleaning — freMTPL2freq

Le nettoyage du dataset assurance a volontairement été limité
aux transformations pouvant être justifiées.

## Transformations réalisées

- conversion de `IDpol` en entier ;
- normalisation des chaînes catégorielles ;
- restriction de la population de modélisation à `0 < Exposure <= 1`.

1 224 observations ont été exclues à cause de cette règle,
soit environ 0,18 % du dataset.

L'occurrence et la fréquence globales restent très proches
avant et après cette opération.

## Valeurs volontairement conservées

Les valeurs extrêmes suivantes n'ont pas été supprimées :

- ClaimNb élevé ;
- VehAge élevé ;
- DrivAge élevé ;
- BonusMalus élevé ;
- Density élevée.

Une valeur extrême n'est pas automatiquement considérée comme une erreur.

Certains traitements statistiques éventuels seront évalués
ultérieurement dans le pipeline de feature engineering ou de modélisation.

# Bilan Cleaning — Vehicle Claim Fraud

Aucune observation complète n'a été supprimée.

## Transformations réalisées

- normalisation des chaînes ;
- `Age = 0` transformé en valeur manquante ;
- `MonthClaimed = "0"` transformé en valeur manquante ;
- `DayOfWeekClaimed = "0"` transformé en valeur manquante ;
- correction de certaines incohérences orthographiques
  dans les noms de marques.

## Décisions conservatrices

Les catégories `"none"` n'ont pas été automatiquement transformées
en valeurs manquantes car leur signification peut varier selon la variable.

Les variables potentiellement sujettes au data leakage ne sont pas encore
supprimées : leur disponibilité au moment de la prédiction devra d'abord
être déterminée.

Les identifiants sont conservés dans le dataset nettoyé mais seront
exclus des features Machine Learning lorsque cela sera approprié.

# Conclusion du Data Cleaning

Les datasets nettoyés ont été enregistrés dans `data/interim/`.

Le nettoyage a privilégié une approche conservatrice :

- correction des incohérences clairement identifiables ;
- absence de suppression arbitraire des outliers ;
- conservation des données originales dans `raw/` ;
- absence d'imputation calculée sur l'ensemble du dataset ;
- traçabilité des décisions de nettoyage.

Les transformations liées à la représentation des variables seront
réalisées lors de l'étape suivante.

## Étape suivante

`04_feature_engineering.ipynb`

Cette étape déterminera :

- les variables utilisées par chaque problème ML ;
- les variables exclues ;
- les features dérivées ;
- les transformations logarithmiques ;
- les représentations temporelles ;
- les variables catégorielles, ordinales et numériques ;
- les éventuelles interactions ;
- les risques de data leakage ;
- la préparation de X et y.

L'encodage, l'imputation et la standardisation définitifs seront intégrés
dans des pipelines scikit-learn afin d'éviter le data leakage.